# Demo 2 · Rapid UI Prototyping: a Customer Support Triage Dashboard 🎨
**The business problem:** a support team gets hundreds of emails a day. Which ones are urgent?

**Today:** we describe a dashboard in English and the AI draws it in seconds. Tools like **v0.dev** do exactly this. Here we do it with Groq so everyone can follow along.

**Plan:** describe the screen → see it → improve it → make the "Run AI" button actually think.

## ⚙️ Setup (run once)
Click the ▶ button on the left of the cell below. When a box appears, **paste your Groq API key** and press **Enter**.

> 🔑 **No key yet?** Go to [console.groq.com/keys](https://console.groq.com/keys) → sign in with Google → **Create API Key** → copy it.
> Treat it like a password. Do not share it or put it in slides.

In [ ]:
# This cell installs the AI library and connects to Groq. You do not need to understand it.
%pip install -q groq
import re, json, html
from getpass import getpass
from groq import Groq
from IPython.display import HTML, display, Markdown

client = Groq(api_key=getpass("Paste your Groq API key and press Enter: "))
MODEL = "openai/gpt-oss-20b"   # fast, free-tier friendly. You can also try "openai/gpt-oss-120b"

def ask_ai(prompt, system="You are a friendly, helpful assistant."):
    """Send a message to the AI and get its answer back as text."""
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
    )
    return reply.choices[0].message.content

def clean_code(text):
    """AI often wraps code in ``` marks. This keeps only the code inside."""
    m = re.search(r"```(?:[a-zA-Z]*)\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def show_page(page, height=480):
    """Show a web page right here inside the notebook."""
    display(HTML(f'<iframe srcdoc="{html.escape(page)}" width="100%" height="{height}" '
                 f'style="border:1px solid #ccc;border-radius:8px"></iframe>'))

print(ask_ai("Reply with exactly: ✅ Connected! The AI is ready.", system="Follow the instruction exactly."))

## Step 1 · Describe the screen (your "spec") 📝
A good description says **what is on the screen** and **how it should feel**. Read it, then run.

In [ ]:
DESIGNER = "You are an expert web designer. Reply with ONLY one complete HTML file (CSS and JavaScript inside it). No explanations."

spec = """
Create a 'Customer Support Triage Dashboard' web page.
- A text box where the agent pastes a customer email.
- A 'Priority' dropdown with: High, Medium, Low.
- A big 'Run AI' button.
- An empty 'AI Result' card below the button.
- Clean, modern, professional look. Soft colours. Rounded corners.
"""

dashboard = clean_code(ask_ai(spec, system=DESIGNER))
show_page(dashboard, height=620)

✅ A working screen in seconds. In class, paste the same spec into **v0.dev** to get a React version.

## Step 2 · Give design feedback 🔁
Directors do not accept the first draft. Change the feedback text and run again as many times as you like.

In [ ]:
feedback = "Add a top bar with the company name 'HelpHub', make the Run AI button orange, and add 3 sample customer emails as small clickable chips that fill the text box."

dashboard = clean_code(ask_ai(f"Here is a web page:\n{dashboard}\n\nImprove it: {feedback}\nReturn the full updated HTML.", system=DESIGNER))
show_page(dashboard, height=650)

## Step 3 · Make "Run AI" actually think 🧠
Right now the button is only a picture of a button. A **prototype** shows the idea. Now we give it a brain: the AI reads an email and decides the priority.

In [ ]:
TRIAGE_RULES = """You are a customer support triage assistant.
Read the customer email and answer ONLY with JSON in this exact shape:
{"priority": "High" or "Medium" or "Low",
 "category": "Billing" or "Technical" or "Delivery" or "Account" or "Other",
 "summary": "one short sentence",
 "reply": "a short, polite reply to the customer (2-3 sentences)"}"""

def triage(email):
    answer = ask_ai(email, system=TRIAGE_RULES)
    return json.loads(re.search(r"\{.*\}", answer, re.S).group(0))

result = triage("Hi, I was charged TWICE for order #4521 and nobody answers my calls. Refund me today!")
result

## Step 4 · Triage a whole inbox at once 📥
Three very different emails. Watch the AI sort them.

In [ ]:
import pandas as pd

inbox = [
    "Hi, I was charged TWICE for order #4521 and nobody answers my calls. Refund me today!",
    "My parcel tracking has not moved for 5 days. When will my lamp arrive?",
    "Hi team, do you have a dark mode in the app? Thanks!",
]
table = pd.DataFrame([triage(e) for e in inbox])
table.insert(0, "email", [e[:45] + "..." for e in inbox])
table

## 🧠 What just happened
- **Prototype** (Steps 1-2): what it *looks* like. Fast and cheap to change.
- **Brain** (Steps 3-4): what it *does*. Same AI, different instruction.

👉 **Next demo:** we join the screen and the brain into a real app that anyone can open with a link.